In [2]:
# ============================================================
# CodeAlpha AI Internship - Language Translation Tool
# Google Translate (via deep-translator) with MyMemory backup + Gradio UI
# Features: auto-detect, swap languages, text-to-speech, error handling
# ============================================================

!pip install -q gradio deep-translator gTTS langdetect

import time
import tempfile
import gradio as gr
from deep_translator import GoogleTranslator, MyMemoryTranslator
from gtts import gTTS
from langdetect import detect

# ---------- 1. Language list (name -> code) ----------
supported = GoogleTranslator().get_supported_languages(as_dict=True)   # {'urdu': 'ur', ...}
LANG_CODES = {name.title(): code for name, code in supported.items()}
CODE_TO_NAME = {code: name for name, code in supported.items()}
TARGET_LANGS = sorted(LANG_CODES)
SOURCE_LANGS = ["Auto Detect"] + TARGET_LANGS
LANG_CODES["Auto Detect"] = "auto"

# ---------- 2. Translation with retry + backup service ----------
def translate_with_fallback(text, source, target):
    src, tgt = LANG_CODES[source], LANG_CODES[target]
    google_error = None

    # Try Google Translate twice (rate limits are often temporary)
    for _ in range(2):
        try:
            return GoogleTranslator(source=src, target=tgt).translate(text)
        except Exception as e:
            google_error = e
            time.sleep(2)

    # Backup: MyMemory (free, no API key, 500 characters max per request)
    try:
        if src == "auto":
            src = detect(text)
        src_name = CODE_TO_NAME.get(src, "english")
        tgt_name = CODE_TO_NAME[tgt]
        return MyMemoryTranslator(source=src_name, target=tgt_name).translate(text)
    except Exception as backup_error:
        raise Exception(f"Google: {google_error} | Backup: {backup_error}")

# ---------- 3. Translate + text-to-speech ----------
def translate_text(text, source, target, speak):
    if not text or not text.strip():
        return "Please enter some text to translate.", None

    try:
        translated = translate_with_fallback(text, source, target)
    except Exception as e:
        return f"Translation failed. Please wait a minute and try again.\n\nDetails: {e}", None

    audio_path = None
    if speak:
        try:
            tts = gTTS(translated, lang=LANG_CODES[target])
            tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
            tts.save(tmp.name)
            audio_path = tmp.name
        except Exception:
            pass   # some languages are not supported for speech

    return translated, audio_path

def swap_languages(source, target, input_text, output_text):
    if source == "Auto Detect":
        return source, target, input_text, output_text   # nothing to swap
    return target, source, output_text, input_text

# ---------- 4. User interface ----------
with gr.Blocks(title="Language Translation Tool") as app:
    gr.Markdown("# 🌐 Language Translation Tool\nCodeAlpha AI Internship  |  Powered by Google Translate")

    with gr.Row():
        source = gr.Dropdown(SOURCE_LANGS, value="Auto Detect", label="From")
        swap_btn = gr.Button("⇄ Swap", scale=0)
        target = gr.Dropdown(TARGET_LANGS, value="Urdu", label="To")

    with gr.Row():
        input_text = gr.Textbox(label="Enter text", lines=6, placeholder="Type or paste text here...")
        output_text = gr.Textbox(label="Translation", lines=6, interactive=False)

    speak = gr.Checkbox(label="🔊 Read the translation aloud", value=False)
    audio_out = gr.Audio(label="Audio", type="filepath")

    translate_btn = gr.Button("Translate", variant="primary")

    translate_btn.click(translate_text, [input_text, source, target, speak], [output_text, audio_out])
    swap_btn.click(swap_languages, [source, target, input_text, output_text],
                   [source, target, input_text, output_text])

    gr.Examples(
        examples=[["Hello, how are you today?", "English", "Urdu", False],
                  ["Machine learning is changing the world.", "English", "French", True],
                  ["Bonjour tout le monde", "Auto Detect", "English", False]],
        inputs=[input_text, source, target, speak],
    )

app.launch(share=True)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2288d1a8cf951c2147.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
